[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mateus-Gabardo/bend2-verification-impl/blob/main/colab.ipynb)

# Verificação de propriedades em Bend 2: reserva de estoque

Este notebook reproduz o estudo de caso do artigo *Características Funcionais na Verificação de Propriedades em Bend 2* (UTFPR, SE4FP/CBSoft 2026).

- **v0** é a versão de referência da função `reserve`.
- **v1** foi alterada com auxílio de IA e tem um erro no caso de produto esgotado.

As duas versões são verificadas contra as mesmas leis:

- **P1–P5:** propriedades gerais, que valem para qualquer entrada.
- **T1–T5:** testes unitários, cada um com uma entrada fixa.

Quem verifica é o próprio compilador do Bend. Ao final de cada arquivo ele imprime:

- `ALL PROOFS CHECK`: todas as leis do arquivo foram provadas;
- `SOME PROOFS FAIL`: alguma lei é falsa ou ficou sem prova.

**Como usar:** execute as células em ordem, de cima para baixo (▶ ou Ctrl+Enter). Nada precisa ser instalado no seu computador, pois tudo roda numa máquina do Google que é apagada quando a sessão fecha.

Repositórios:
[código](https://github.com/Mateus-Gabardo/bend2-verification-impl) ·
[artigo](https://github.com/Mateus-Gabardo/bend2-verification)

## 1. Instalar o Bend 2

O comando abaixo baixa o script oficial de instalação e o executa.

- O `!` no início da linha faz o Colab rodar o comando no terminal Linux, e não como Python.
- O binário fica em `~/.bend/bin/bend`.
- A segunda parte da célula, em Python, coloca essa pasta no `PATH`, para que as próximas células possam chamar só `bend`.

O estudo usou a versão **2.0.40**. Se a versão instalada for outra, os resultados podem mudar.

In [ ]:
!curl -fsSL https://bend-lang.com/install.sh | sh

import os
os.environ["PATH"] += ":" + os.path.expanduser("~/.bend/bin")

!bend version

## 2. Copiar o repositório para o Colab

O `git clone` copia o repositório público do GitHub para a máquina do Colab. Não é preciso baixar nada antes.

O `%cd` entra na pasta copiada. A partir daqui, todos os comandos rodam dentro dela. O ícone de pasta 📁, no menu da esquerda, mostra os arquivos.

Se a célula for executada de novo, o `rm -rf` apaga a cópia anterior antes de clonar outra vez.

In [ ]:
%cd /content
!rm -rf bend2-verification-impl
!git clone https://github.com/Mateus-Gabardo/bend2-verification-impl
%cd /content/bend2-verification-impl
!ls v0 v1

## 3. O programa: reservar unidades de um produto

O estoque é uma lista de produtos. Cada produto é `Product{id, disponível, reservado}`.

`reserve(estoque, id, quantidade)` tem dois resultados possíveis:

- **`Accepted{novo estoque}`:** a quantidade sai do *disponível* e entra no *reservado*;
- **`Refused{estoque}`:** o pedido é recusado e o estoque fica igual. Isso acontece com quantidade zero, produto inexistente ou quantidade maior que a disponível.

A célula abaixo mostra a função principal e roda o exemplo de `main.bend`, que reserva 3 unidades do produto 7 (disponível 10, reservado 2).

Resultado esperado: o produto 7 fica com disponível **7** e reservado **5**.

In [ ]:
!cat v0/src/reserve.bend
!echo "----- executando v0/main.bend -----"
!cd v0 && bend main.bend

## 4. O que a IA mudou na v1

O `diff` compara a busca do produto (`src/search.bend`) nas duas versões. Linhas com `<` são da v0 e linhas com `>` são da v1.

**O erro:** a v1 passou a usar `keep_if_stocked`, que remove o produto da lista quando o disponível chega a zero. Com isso, as unidades que acabaram de ser reservadas também somem.

O restante do código é igual nas duas versões.

In [ ]:
!diff v0/src/search.bend v1/src/search.bend

## 5. As leis

`LAWS.bend` é **o mesmo arquivo** nas duas versões.

- Cada `law` declara uma igualdade que precisa valer.
- Nas propriedades P1–P5, `for` significa *para todo*. A lei vale para qualquer lista, produto e quantidade.
- Nos testes T1–T5 não há `for`, porque cada um usa uma única entrada fixa.

Para ver como cada lei é provada, abra `PROOF.bend` (`def Laws.<nome_da_lei>`).

In [ ]:
!cat v0/LAWS.bend

## 6. v0: todas as leis são provadas

`PROOF.bend` importa as leis e traz uma prova para cada uma.

O compilador confere cada passo da prova. Se todas fecharem, ele imprime `ALL PROOFS CHECK`.

**Esperado:** `ALL PROOFS CHECK`.

In [ ]:
!cd v0 && bend PROOF.bend

## 7. v1: os testes unitários continuam passando

`unit_tests.bend` verifica apenas T1–T5.

Nenhum desses testes pede **todo** o disponível de um produto. Por isso, o erro da v1 nunca aparece neles.

**Esperado:** `ALL PROOFS CHECK`, ou seja, os testes **não** detectam o erro.

In [ ]:
!cd v1 && bend unit_tests.bend

## 8. v1: o caso esgotado viola P1 (atualização)

`sold_out_p1.bend` pede as 3 unidades restantes do produto 7, ou seja, todo o disponível.

Pela P1, o produto 7 deveria ficar com disponível 0 e reservado 5. A célula roda o mesmo arquivo nas duas versões:

- **v0:** `ALL PROOFS CHECK`.
- **v1:** `SOME PROOFS FAIL`.

Na saída de erro da v1:

- `expected` é o lado esquerdo da igualdade, que é o que a v1 de fato calculou: o produto 7 sumiu da lista.
- `observed` é o lado direito, que é o resultado correto.

In [ ]:
!cat v1/sold_out_p1.bend
!echo "===== v0 ====="
!cd v0 && bend sold_out_p1.bend
!echo "===== v1 ====="
!cd v1 && bend sold_out_p1.bend

## 9. v1: o caso esgotado viola P2 (conservação)

A P2 diz que a soma *disponível + reservado* de todo o estoque não muda com uma reserva.

No exemplo, a soma antes da reserva é 3+2 (produto 7) + 5+0 (produto 8) = **10**.

- **v0:** `ALL PROOFS CHECK`, pois a soma continua 10.
- **v1:** `SOME PROOFS FAIL`, pois a soma cai para **5**. As 5 unidades do produto 7 sumiram junto com ele.

In [ ]:
!echo "===== v0 ====="
!cd v0 && bend sold_out_p2.bend
!echo "===== v1 ====="
!cd v1 && bend sold_out_p2.bend

## 10. O ponto cego dos testes

`blind_spot.bend` prova que, **sempre que o produto não esgota**, a v1 dá exatamente o mesmo resultado que a v0.

Consequência: nenhum teste unitário que não esgote o produto consegue distinguir as duas versões. Só uma lei geral, como P1 ou P2, cobre o caso problemático.

**Esperado:** `ALL PROOFS CHECK` nas duas versões.

In [ ]:
!echo "===== v0 ====="
!cd v0 && bend blind_spot.bend
!echo "===== v1 ====="
!cd v1 && bend blind_spot.bend

## 11. v1: as provas não fecham

Na v1, P1 e P2 são falsas, portanto não existe prova para elas. Em `v1/PROOF.bend` elas ficaram como `TODO` (prova incompleta), e o compilador recusa o arquivo.

P3, P4 e P5 continuam provadas, com as mesmas provas da v0.

**Esperado:** `SOME PROOFS FAIL` e `2 TODOs found`.

In [ ]:
!cd v1 && bend PROOF.bend

## 12. Rodar tudo de uma vez

`run_all.sh` executa todas as verificações nas duas versões e grava as saídas em `logs/`.

Ele também aplica as provas da v0, sem nenhuma adaptação, à v1. Essas provas falham.

A tabela final resume o resultado de cada arquivo.

In [ ]:
!sh run_all.sh

## 13. Experimente: um teste que pega o erro

A célula abaixo cria o arquivo `t6.bend`, com um sexto teste unitário que pede **todas** as 10 unidades do produto 7.

Pela v0, o resultado correto é o produto 7 com disponível 0 e reservado 12. O arquivo é copiado para as duas pastas e verificado em cada uma:

- **v0:** `ALL PROOFS CHECK`.
- **v1:** `SOME PROOFS FAIL`, pois o produto 7 some da lista.

Isso mostra que um teste unitário *pode* pegar o erro, mas só se alguém pensar exatamente nesse caso. A lei P1 cobre todos os casos sem que ninguém precise lembrar dele.

In [ ]:
%%writefile t6.bend
# T6: pede todo o disponível do produto 7 (o caso esgotado)
import Base
import ./src/stock.bend as Stock
import ./src/reserve.bend as Res
import ./SPEC.bend as Spec

law t6_sold_out:
  {Res.reserve(Spec.sample(), 7n, 10n)
    == Stock.Accepted{[Stock.Product{7n, 0n, 12n}, Stock.Product{8n, 5n, 0n}]} : Stock.Outcome}

def t6_sold_out():
  {==}

In [ ]:
!cp t6.bend v0/ && cp t6.bend v1/
!echo "===== v0 ====="
!cd v0 && bend t6.bend
!echo "===== v1 ====="
!cd v1 && bend t6.bend

## 14. Experimente: corrigir a v1

A célula abaixo desfaz a alteração da IA: troca `Some{keep_if_stocked(p2, rest)}` de volta por `Some{p2 <> rest}` em `v1/src/search.bend`. Em seguida, roda de novo os casos que falhavam.

**Esperado:** `ALL PROOFS CHECK` em `sold_out_p1`, `sold_out_p2` e `t6`.

Você também pode editar os arquivos à mão: abra pelo 📁 com dois cliques e salve com Ctrl+S. As alterações valem só nesta sessão e **não** voltam para o GitHub.

In [ ]:
!sed -i 's/Some{keep_if_stocked(p2, rest)}/Some{p2 <> rest}/' v1/src/search.bend
!grep -n "Some{p2" v1/src/search.bend
!cd v1 && bend sold_out_p1.bend
!cd v1 && bend sold_out_p2.bend
!cd v1 && bend t6.bend